# Qwen Image 2.1 — Setup

Notebook pensato per **Google Colab con A100**.

**Nuova sessione Colab:**
1. seleziona una GPU A100 (e RAM elevata se disponibile);
2. esegui la cella **Installazione**;
3. fai **Runtime → Riavvia sessione**;
4. riparti dalla cella **Carica Qwen Image 2.1**.

Non aggiornare manualmente Torch/CUDA.

In [ ]:
# Controllo GPU (opzionale)
!nvidia-smi

In [ ]:
# Installazione librerie
!pip install -q -U "transformers>=5.17,<5.18" accelerate pillow
!pip install -q -U git+https://github.com/huggingface/diffusers.git
!pip uninstall -y torchao

print("✅ Installazione completata.")
print("Ora: Runtime → Riavvia sessione, poi riparti dalla cella successiva.")

## Carica Qwen Image 2.1

Esegui questa cella **dopo il riavvio della sessione**.

In [ ]:
import torch
from diffusers import QwenImage21Pipeline

pipe = QwenImage21Pipeline.from_pretrained(
    "Qwen/Qwen-Image-2.1",
    torch_dtype=torch.bfloat16
).to("cuda")

print("✅ Pipeline caricata su GPU")
print("GPU:", torch.cuda.get_device_name(0))

# Image Edit Batch

Metti le immagini da modificare in:

`MyDrive/Qwen/input`

Ogni esecuzione del batch crea automaticamente una nuova cartella timestamp dentro:

`MyDrive/Qwen/output`

In [ ]:
# Collega Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cartelle di lavoro
import os

BASE_DIR = "/content/drive/MyDrive/Qwen"
INPUT_DIR = os.path.join(BASE_DIR, "input")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("📥 Input:", INPUT_DIR)
print("📤 Output:", OUTPUT_DIR)

## Prompt e qualità

- `STEPS = 20`: veloce, ottimo per prove.
- `STEPS = 30`: compromesso qualità/tempo.
- `STEPS = 40`: valore raccomandato/default di Qwen, ideale per finali.
- `OUTPUT_RESOLUTION = 1024`: veloce; mantiene automaticamente il rapporto dell'immagine.
- `OUTPUT_RESOLUTION = 2048`: qualità finale 2K, più lenta.
- `SEED`: lascia 42 per risultati ripetibili; cambialo per esplorare varianti.

In [ ]:
# Impostazioni
PROMPT = '''
Place the product in an elegant luxury wellness center.

Modern premium architecture, warm ambient lighting,
photorealistic commercial photography.

Keep the product extremely consistent with the original image:
same shape, proportions, colors, materials and details.
Do not redesign or modify the product itself.

Premium catalog photography, realistic lighting, high detail.
'''

STEPS = 20
SEED = 42
OUTPUT_RESOLUTION = 1024   # prova 2048 per i finali

print("✅ Impostazioni pronte")
print("Steps:", STEPS, "| Seed:", SEED, "| Resolution:", OUTPUT_RESOLUTION)

In [ ]:
# Batch automatico — una nuova cartella per ogni run
import os
import json
import torch
from PIL import Image
from datetime import datetime

if "pipe" not in globals():
    raise RuntimeError("❌ Devi prima caricare Qwen Image 2.1.")
if "PROMPT" not in globals():
    raise RuntimeError("❌ Devi prima eseguire la cella Prompt e qualità.")

STEPS = globals().get("STEPS", 20)
SEED = globals().get("SEED", 42)
OUTPUT_RESOLUTION = globals().get("OUTPUT_RESOLUTION", 1024)

extensions = (".png", ".jpg", ".jpeg", ".webp")
files_to_process = sorted(
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith(extensions)
)

if not files_to_process:
    print("⚠️ Nessuna immagine trovata nella cartella input.")
else:
    run_id = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    RUN_OUTPUT_DIR = os.path.join(OUTPUT_DIR, run_id)
    os.makedirs(RUN_OUTPUT_DIR, exist_ok=True)

    # Salva prompt e impostazioni per ricordare come è stato creato il batch
    with open(os.path.join(RUN_OUTPUT_DIR, "prompt.txt"), "w", encoding="utf-8") as f:
        f.write(PROMPT.strip())

    with open(os.path.join(RUN_OUTPUT_DIR, "settings.json"), "w", encoding="utf-8") as f:
        json.dump({
            "steps": STEPS,
            "seed": SEED,
            "output_resolution": OUTPUT_RESOLUTION,
        }, f, indent=2)

    print(f"📁 Trovate {len(files_to_process)} immagini.")
    print(f"📂 Output: {RUN_OUTPUT_DIR}\n")

    for i, filename in enumerate(files_to_process, start=1):
        input_path = os.path.join(INPUT_DIR, filename)
        name, _ = os.path.splitext(filename)
        output_path = os.path.join(RUN_OUTPUT_DIR, f"{name}_QWEN.png")

        print(f"[{i}/{len(files_to_process)}] 🎨 {filename}")

        try:
            input_image = Image.open(input_path).convert("RGB")

            result = pipe(
                prompt=PROMPT,
                image=input_image,
                num_inference_steps=STEPS,
                output_resolution=OUTPUT_RESOLUTION,
                generator=torch.Generator("cuda").manual_seed(SEED),
            ).images[0]

            result.save(output_path)
            print(f"    ✅ {name}_QWEN.png")

        except Exception as e:
            print(f"    ❌ Errore: {e}")

    print("\n🎉 Batch terminato.")